# Logistic Regression Fraud Detection Experiment
IEEE-CIS Fraud Detection — Underfit Baseline Analysis

## 0. Setup & MLflow Configuration

In [ ]:
import os, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from kaggle_secrets import UserSecretsClient

DAGSHUB_TOKEN = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
DAGSHUB_USERNAME = 'delibaaa'
REPO_NAME        = 'DELIBA-ML-ASSIGNMENT-2'
os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN

mlflow.set_tracking_uri(f'https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow')
mlflow.set_experiment('LogisticRegression_Training')
print('Setup complete')

## 1. Data Loading

In [ ]:
DATA_PATH = '/kaggle/input/ieee-fraud-detection'
train = pd.read_csv(f'{DATA_PATH}/train_transaction.csv').merge(
        pd.read_csv(f'{DATA_PATH}/train_identity.csv'), on='TransactionID', how='left')
test  = pd.read_csv(f'{DATA_PATH}/test_transaction.csv').merge(
        pd.read_csv(f'{DATA_PATH}/test_identity.csv'), on='TransactionID', how='left')
print(f'Train: {train.shape} | Fraud rate: {train["isFraud"].mean():.4f}')

## 2. Cleaning

In [ ]:
with mlflow.start_run(run_name='LogisticRegression_Cleaning'):
    nan_pct = train.isnull().mean()
    high_nan_cols = nan_pct[nan_pct > 0.9].index.tolist()
    train_clean = train.drop(columns=high_nan_cols)
    test_clean  = test.drop(columns=high_nan_cols)

    constant_cols = [c for c in train_clean.columns if train_clean[c].nunique(dropna=False) <= 1]
    train_clean = train_clean.drop(columns=constant_cols)
    test_clean  = test_clean.drop(columns=constant_cols, errors='ignore')

    mlflow.log_param('nan_strategy', 'median_impute_then_standardscale')
    mlflow.log_param('dropped_high_nan_cols', len(high_nan_cols))
    mlflow.log_param('dropped_constant_cols', len(constant_cols))
    mlflow.log_metric('cols_after_cleaning', train_clean.shape[1])
    print(f'After cleaning: {train_clean.shape}')

## 3. Feature Engineering

In [ ]:
with mlflow.start_run(run_name='LogisticRegression_Feature_Engineering'):

    def feature_engineering(df):
        df = df.copy()
        df['TransactionHour']    = (df['TransactionDT'] // 3600 % 24).astype(int)
        df['TransactionDay']     = (df['TransactionDT'] // (3600*24)).astype(int)
        df['TransactionWeekDay'] = (df['TransactionDay'] % 7).astype(int)
        df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
        for col in ['card1', 'card2', 'addr1']:
            if col in df.columns:
                df[f'{col}_freq'] = df[col].map(df[col].value_counts(normalize=True)).fillna(0)
        return df

    train_fe = feature_engineering(train_clean)
    test_fe  = feature_engineering(test_clean)

    mlflow.log_metric('features_after_fe', train_fe.shape[1])
    print(f'After FE: {train_fe.shape}')

## 4. Feature Selection

In [ ]:
with mlflow.start_run(run_name='LogisticRegression_Feature_Selection'):

    TARGET = 'isFraud'
    DROP_COLS = ['TransactionID', 'TransactionDT', TARGET]
    feature_cols = [c for c in train_fe.columns if c not in DROP_COLS]

    cat_cols = train_fe[feature_cols].select_dtypes(include='object').columns.tolist()
    num_cols = train_fe[feature_cols].select_dtypes(exclude='object').columns.tolist()

    X_temp = train_fe[num_cols].fillna(train_fe[num_cols].median())
    corr_matrix = X_temp.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.95)]
    num_cols_filtered = [c for c in num_cols if c not in high_corr_cols]

    selected_features = num_cols_filtered[:50] + cat_cols[:20]

    mlflow.log_param('selected_features_count', len(selected_features))
    mlflow.log_param('selection_method', 'correlation_filter_top50_numeric_top20_cat')
    mlflow.log_param('high_corr_dropped', len(high_corr_cols))
    print(f'Selected {len(selected_features)} features')

## 5. Cross Validation — Different C Values (Regularization Analysis)

In [ ]:
TARGET = 'isFraud'
X_raw = train_fe[selected_features].copy()
y = train_fe[TARGET]
cat_in_sel = [c for c in selected_features if train_fe[c].dtype == 'object']
num_in_sel = [c for c in selected_features if train_fe[c].dtype != 'object']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), num_in_sel),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), cat_in_sel)
])

C_values = [0.001, 0.01, 0.1, 1.0, 10.0]

for C in C_values:
    with mlflow.start_run(run_name=f'LogisticRegression_C_{C}'):
        pipe = Pipeline([
            ('pre', preprocessor),
            ('model', LogisticRegression(C=C, max_iter=1000, class_weight='balanced',
                                         solver='lbfgs', random_state=42, n_jobs=-1))
        ])
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        cv_scores = cross_val_score(pipe, X_raw, y, cv=skf, scoring='roc_auc')

        print(f'C={C:6.3f} | CV AUC: {cv_scores.mean():.5f} (+/- {cv_scores.std():.5f})')
        mlflow.log_param('C', C)
        mlflow.log_param('solver', 'lbfgs')
        mlflow.log_param('class_weight', 'balanced')
        mlflow.log_metric('cv_auc_mean', cv_scores.mean())
        mlflow.log_metric('cv_auc_std', cv_scores.std())

## 6. Final Pipeline Training & Registration

In [ ]:
BEST_C = 1.0

with mlflow.start_run(run_name='LogisticRegression_Final') as run:

    X_raw = train_fe[selected_features].copy()
    y_final = train_fe[TARGET]
    cat_in_sel = [c for c in selected_features if train_fe[c].dtype == 'object']
    num_in_sel = [c for c in selected_features if train_fe[c].dtype != 'object']

    preprocessor = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scale', StandardScaler())
        ]), num_in_sel),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
            ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), cat_in_sel)
    ])

    final_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(C=BEST_C, max_iter=1000, class_weight='balanced',
                                      solver='lbfgs', random_state=42, n_jobs=-1))
    ])

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    final_cv = cross_val_score(final_pipeline, X_raw, y_final, cv=skf, scoring='roc_auc', n_jobs=-1)
    print(f'Final LR Pipeline CV AUC: {final_cv.mean():.5f} (+/- {final_cv.std():.5f})')

    final_pipeline.fit(X_raw, y_final)

    mlflow.log_param('C', BEST_C)
    mlflow.log_metric('final_cv_auc_mean', final_cv.mean())
    mlflow.log_metric('final_cv_auc_std', final_cv.std())

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path='lr_pipeline',
        registered_model_name='LogisticRegression_Fraud_Pipeline'
    )
    print(f'LR pipeline registered! Run ID: {run.info.run_id}')

with open('lr_selected_features.json', 'w') as f:
    json.dump(selected_features, f)